In [1]:
!pip install -q accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01


In [2]:
!rm -rf ~/.cache/huggingface/hub

In [2]:
import pandas as pd
import numpy as np
import random
import os
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [4]:
# HUGGING FACE LOGIN (Required for Llama 3)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [23]:
# LOAD LLAMA-3-8B LOCALLY (with 4-bit Quantization)
print("Loading Llama-3-8B-Instruct...")
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 1. Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 2. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

# 3. Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

print("Model loaded successfully!")

Loading Llama-3-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully!


In [7]:
# LOAD DATA
file_path = '/kaggle/input/datasets/noor3027/clinical-data/Clinical_and_Other_Features.xlsx'
try:
    df_raw = pd.read_excel(file_path, header=None)
    data = df_raw.iloc[3:].copy()
except FileNotFoundError:
    print("File not found. Using dummy data for testing.")
    data = pd.DataFrame({0: ['Patient_1'], 36: ['L'], 37: ['10 oclock'], 69: ['1'], 70: ['1'], 71: ['2'], 75: ['2.5'], 50: ['1'], 48: ['0'], 51: ['0'], 52: ['0'], 49: ['0'], 72: ['0'], 80: ['0'], 81: ['0']})

In [8]:
# DEFINE MAP
shape_map = {"0": "oval", "1": "irregular", "2": "lobular", "3": "reniform", "4": "stellate"}
margin_map = {"0": "obscured", "1": "ill-defined", "2": "spiculated", "3": "indistinct", "4": "circumscribed"}
density_map = {"0": "heterogeneous", "1": "scattered", "2": "minimal", "3": "moderate", "4": "extremely", "5": "predominantly fatty"}
echo_map = {"0": "hypoechoic", "1": "hyperechoic", "2": "isoechoic", "3": "anechoic", "4": "irregular", "5": "mixed", "6": "boundary"}
calc_map = {"0": "pleomorphic", "1": "present (unspecified)", "2": "heterogeneous", "3": "microcalcifications", "4": "linear", "5": "clustered", "6": "amorphous", "7": "branching"}
yes_no_map = {"0": "no", "1": "yes"}
side_map = {"L": "left", "R": "right"}

In [9]:
# CLEANING FUNCTION
def clean_val(val, mapping=None):
    if pd.isna(val):
        return None
    val_str = str(val).strip().upper()
    if val_str in ['NC', 'NA', 'NP', 'NAN', 'NONE', '']:
        return None
    
    clean_key = str(val).strip()
    if clean_key.endswith('.0'):
        clean_key = clean_key[:-2]
        
    if mapping:
        return mapping.get(clean_key, clean_key) # fallback to key if not in map
    return clean_key

In [10]:
# EXTRACT DATA COLUMNS
df = pd.DataFrame({
    'patient_id': data[0],
    'laterality': data[36].apply(lambda x: clean_val(x, side_map)), 
    'position': data[37].apply(lambda x: clean_val(x)),
    'density': data[69].apply(lambda x: clean_val(x, density_map)),
    
    # Mammography Features
    'm_shape': data[70].apply(lambda x: clean_val(x, shape_map)),
    'm_margin': data[71].apply(lambda x: clean_val(x, margin_map)),
    'm_size': data[75].apply(lambda x: clean_val(x)),
    'arch_dist': data[72].apply(lambda x: clean_val(x, yes_no_map)), # Architectural distortion
    'calcifications': data[74].apply(lambda x: clean_val(x, calc_map)), # Calcifications
    
    # Ultrasound Features
    'us_shape': data[76].apply(lambda x: clean_val(x, shape_map)),
    'us_margin': data[77].apply(lambda x: clean_val(x, margin_map)),
    'us_size': data[78].apply(lambda x: clean_val(x)),
    'us_echo': data[79].apply(lambda x: clean_val(x, echo_map)),
    'us_solid': data[80].apply(lambda x: clean_val(x, yes_no_map)), # Solid mass
    'us_shadow': data[81].apply(lambda x: clean_val(x, yes_no_map)), # Posterior shadowing
    
    # Associated/MRI Findings
    'nodes': data[50].apply(lambda x: clean_val(x, yes_no_map)),
    'multifocal': data[48].apply(lambda x: clean_val(x, yes_no_map)),
    'skin_inv': data[51].apply(lambda x: clean_val(x, yes_no_map)),
    'pec_inv': data[52].apply(lambda x: clean_val(x, yes_no_map)),
    'contralateral': data[49].apply(lambda x: clean_val(x, yes_no_map)) # Contralateral involvement
})

In [27]:
# LLAMA-3-SPECIFIC PROMPT GENERATOR
def create_medical_prompt(row):
    findings = []
    
    if row['density']: findings.append(f"Breast density: {row['density']}.")
    
    # Location
    loc_str = []
    if row['laterality']: 
        loc_str.append(f"{row['laterality']} breast")
    
    if row['position']: 
        pos_raw = str(row['position']).strip().upper()
        
        # If the cell is literally just 'L' or 'R', ignore it (we already have laterality)
        if pos_raw in ['L', 'R']:
            pos_clean = ""
        else:
            # Strip prefixes and format nicely
            pos_clean = pos_raw.replace('L ', '').replace('R ', '').strip()
            
            # Catch plain numbers (e.g., '1' -> "1 o'clock")
            if pos_clean.isdigit():
                pos_clean = f"{pos_clean} o'clock"
            # Catch things like '9 MEDIAL' -> "9 o'clock medial"
            elif pos_clean[0].isdigit() and ' ' in pos_clean:
                parts = pos_clean.split(' ', 1)
                pos_clean = f"{parts[0]} o'clock {parts[1].lower()}"
                
        if pos_clean:
            loc_str.append(f"{pos_clean} position")
            
    if loc_str: findings.append(f"Mass location: {' at '.join(loc_str)}.")
    
    shape = row['m_shape'] or row['us_shape'] 
    if shape: findings.append(f"Lesion shape: {shape}.")
        
    margin = row['m_margin'] or row['us_margin']
    if margin: findings.append(f"Margins: {margin}.")
        
    size = row['m_size'] or row['us_size']
    if size: findings.append(f"Maximum dimension: {size} cm.")
    
    if row['us_echo']: findings.append(f"Echogenicity: {row['us_echo']}.")
    if row['us_solid'] == 'yes': findings.append("Lesion appears solid on ultrasound.")
    if row['us_shadow'] == 'yes': findings.append("Posterior acoustic shadowing is noted.")
    if row['arch_dist'] == 'yes': findings.append("Associated architectural distortion is present.")
    if row['calcifications']: findings.append(f"Associated calcifications: {row['calcifications']}.")
    
    if row['nodes'] == 'yes': findings.append("Suspicious axillary lymphadenopathy present.")
    if row['multifocal'] == 'yes': findings.append("Multicentric/multifocal disease pattern.")
    if row['skin_inv'] == 'yes': findings.append("Skin or nipple involvement noted.")
    if row['pec_inv'] == 'yes': findings.append("Pectoral muscle involvement noted.")
    if row['contralateral'] == 'yes': findings.append("Suspicious findings noted in the contralateral breast.")
    
    findings_str = " ".join(findings)
    if not findings_str: return None
    
    style_hints = [
        "Begin by describing the background breast tissue density before detailing the primary lesion.",
        "Start directly with the primary mass, mentioning background density later.",
        "Use a highly concise, telegraphic medical style.",
        "Use a fluid, narrative clinical style.",
        "Integrate the associated findings into the same sentence as the primary lesion description.",
        "Write in short, punchy, objective declarative sentences."
    ]
    chosen_style = random.choice(style_hints)
    
    # ---> FEW-SHOT PROMPTING FORMATTED FOR LLAMA 3 <---
    messages = [
        {"role": "system", "content": f"You are a strict medical data parser. Convert the descriptors into a short, fluid radiology sentence.\nSTYLE: {chosen_style}\nRULES: Use ONLY provided data. Do not invent features, anatomy, or sizes. VARY YOUR GRAMMAR AND SENTENCE STRUCTURE; avoid repeating the same starting phrases. Do not mention missing data. Output ONLY the clinical text."},
        
        # Example 1: Structure -> "Evaluation reveals..."
        {"role": "user", "content": "Descriptors: Breast density: scattered. Mass location: Left breast at 2 o'clock position. Lesion shape: irregular. Margins: spiculated. Maximum dimension: 1.5 cm. Associated calcifications: pleomorphic.\nReport:"},
        {"role": "assistant", "content": "Evaluation reveals a 1.5 cm irregular, spiculated mass at the 2 o'clock position of the left breast, accompanied by pleomorphic calcifications against a background of scattered fibroglandular density."},
        
        # Example 2: Structure -> "Noted in the [location] is..."
        {"role": "user", "content": "Descriptors: Mass location: Right breast at 10 o'clock position. Suspicious axillary lymphadenopathy present.\nReport:"},
        {"role": "assistant", "content": "Noted in the right breast at the 10 o'clock position is a mass, with associated suspicious ipsilateral axillary lymphadenopathy."},
        
        # Example 3: Structure -> "[Location] imaging demonstrates..." (Sparse data)
        {"role": "user", "content": "Descriptors: Mass location: Left breast.\nReport:"},
        {"role": "assistant", "content": "Left breast imaging demonstrates an abnormal finding."},
        
        # Example 4: Structure -> "A [size] [shape] mass is visualized..."
        {"role": "user", "content": "Descriptors: Breast density: heterogeneously dense. Mass location: Right breast at 12 o'clock position. Lesion shape: oval. Margins: circumscribed. Maximum dimension: 2.0 cm.\nReport:"},
        {"role": "assistant", "content": "A 2.0 cm oval mass with circumscribed margins is visualized in the right breast at the 12 o'clock position. The background tissue is heterogeneously dense."},
        
        # Example 5: Structure -> "There is..." 
        {"role": "user", "content": "Descriptors: Mass location: Left breast at 6 o'clock position. Associated architectural distortion is present. Multicentric/multifocal disease pattern.\nReport:"},
        {"role": "assistant", "content": "There is a mass located in the left breast at the 6 o'clock position, demonstrating a multicentric or multifocal disease pattern with associated architectural distortion."},
        
        # The Actual Patient Data
        {"role": "user", "content": f"Descriptors: {findings_str}\nReport:"}
    ]
    # The tokenizer handles converting this into Llama 3 syntax automatically
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

In [28]:
# LLM GENERATION FUNCTION WITH TAG STRIPPING
def get_llm_report(prompt):
    if not prompt: return "No descriptive imaging data available for this patient."
    try:
        outputs = pipe(prompt)
        report = outputs[0]['generated_text']
        
        # 1. Force strip the Llama 3 prompt template
        if "<|start_header_id|>assistant<|end_header_id|>" in report:
            report = report.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
            
        # 2. Clean up any trailing end-of-turn tokens
        report = report.replace("<|eot_id|>", "").replace("<|end_of_text|>", "").strip()
        
        # 3. Post-processing to chop off chatbot notes (just in case)
        report = report.split('\n\n')[0].strip()
        report = report.split('(')[0].strip()
        report = report.split('Note:')[0].strip()
        
        return report
    except Exception as e:
        print(f"\n⚠️ Inference Error: {e}")
        return "Error generating report."

In [29]:
# RUN TEST
print("Starting report generation for test set...")
test_df = df.head(5).copy()
tqdm.pandas()
test_df['generated_report'] = test_df.progress_apply(lambda row: get_llm_report(create_medical_prompt(row)), axis=1)

print("\n" + "="*50)
print("🩺 TEST RESULTS: GENERATED RADIOLOGY REPORTS")
print("="*50)
for index, row in test_df.iterrows():
    print(f"Patient ID: {row['patient_id']}")
    print(f"Report: {row['generated_report']}")
    print("-" * 50)

Starting report generation for test set...


100%|██████████| 5/5 [00:20<00:00,  4.00s/it]


🩺 TEST RESULTS: GENERATED RADIOLOGY REPORTS
Patient ID: Breast_MRI_001
Report: A mass is visualized in the left breast, situated at the 9 o'clock medial position.
--------------------------------------------------
Patient ID: Breast_MRI_002
Report: A mass is visualized in the left breast at the 1 o'clock position.
--------------------------------------------------
Patient ID: Breast_MRI_003
Report: In the left breast, a mass is seen at the 2 o'clock position, characterized by pleomorphic calcifications, with a scattered breast density background. Additionally, suspicious axillary lymphadenopathy is present, and a multicentric or multifocal disease pattern is noted.
--------------------------------------------------
Patient ID: Breast_MRI_004
Report: The left breast contains an abnormality.
--------------------------------------------------
Patient ID: Breast_MRI_005
Report: In the right breast, a mass is seen at the 3 o'clock position, accompanied by suspicious axillary lymphadenopath

In [30]:
# RUN GENERATION
print("Starting Gemini report generation...")
tqdm.pandas()

df['generated_report'] = df.progress_apply(lambda row: get_llm_report(create_medical_prompt(row)), axis=1)

Starting Gemini report generation...


  0%|          | 0/922 [00:00<?, ?it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 922/922 [1:01:52<00:00,  4.03s/it]


In [31]:
#SAVE TO CSV
output_df = df[['patient_id', 'generated_report']]
output_df.to_csv('llama_patient_radiology_reports.csv', index=False)

print("Success! Generated reports saved to 'patient_radiology_reports.csv'")
output_df.head()

Success! Generated reports saved to 'patient_radiology_reports.csv'


,patient_id,generated_report
3,Breast_MRI_001,A mass is visualized in the medial left breast...
4,Breast_MRI_002,A mass is seen in the left breast at the 1 o'c...
5,Breast_MRI_003,A mass is seen in the 2 o'clock position of th...
6,Breast_MRI_004,An abnormality is seen in the left breast.
7,Breast_MRI_005,A mass is identified in the right breast at th...
